In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4

max_iters = 1000

learning_rate = 3e-4
eval_iters = 250
# dropout = 0.2

cuda


In [2]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)


['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [3]:
string_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_string = { i:ch for i, ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([ 1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,
         0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,
         0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36,
        25, 38, 28,  1, 39, 30,  1, 39, 50,  9])


In [4]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #print(ix)

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x)
print('targets')
print(y)

inputs:
tensor([[60,  1, 55, 58, 72, 62, 57, 58],
        [57, 58,  1, 66, 78,  1, 61, 68],
        [72, 72,  9,  1, 54, 67, 57,  0],
        [ 1, 71, 58, 72, 73, 58, 57,  0]], device='cuda:0')
targets
tensor([[ 1, 55, 58, 72, 62, 57, 58,  1],
        [58,  1, 66, 78,  1, 61, 68, 66],
        [72,  9,  1, 54, 67, 57,  0, 73],
        [71, 58, 72, 73, 58, 57,  0, 74]], device='cuda:0')


In [5]:
@torch.no_grad() # don't use any gradients, reduces computations
def estimate_loss():
    out = {}
    model.eval() # model is being tested, dropout is turned off in this mode
    for split in ['train', 'val']:
        for k in range(eval_iters):
            losses = torch.zeros(eval_iters)
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [6]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape #batch, time, channels
            logits = logits.view(B*T, C) # reshaping for nn functions
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) #loss

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            # get predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample from distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T + 1)

        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
            


,DrFms
Lr[EbtKnW"HmBQw5F0r*e'axsY]4&twA(boW4USb"m6A!W?JqZ(d2NLWCwbX;
NH"Qyp
NOSZceQw
uIR wSKIDbw.k4B1Hn)HJ"" ??8i86w*UZ)UHmpOjJH)3
1mT9e'STO8tlPP1"*y;DNK2f-gJ-LZOE[Os szi3.G8MF)QQfu:G-p1IHxO-zjRiHflg-!Z[_vodR]:C][yy&R]GVMc7q3deST72Vy*ml(gA&xS9l9IRKp:M.Deo ?E1Zvvji,Mlcq2mtQZ:[(WA!j)x
NYW&o?55!VsRx:j:-'e'O8]hU".F9vpx5r469]f_hEVPp!eJBr3pYuyS ,WC'OBhhwLA6VhG&7
8A!]gmwMC_)xJbG!-!lWZnagBB;LT*DoQf2.I,_hDEHDI3HBjGIE-HU2T?TTbzA"U0b:SsE_)3iu3(gUci"(R.,LP1hG;QSz*4kfN:b)

o3TOAYSaF[0beSThyN2NQsvj'KXx*N'oQwY


In [13]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")
    # sample from training data
    xb, yb = get_batch('train')

    #evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())


step: 0, train loss: 0.0112, val loss: 0.0102
step: 250, train loss: 0.0112, val loss: 0.0103
step: 500, train loss: 0.0108, val loss: 0.0086
step: 750, train loss: 0.0112, val loss: 0.0108
2.495784282684326


In [8]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


'2J!l[74cifAdacis, D5qt Ier9cTo2_ovUctzCPwallIQG&m
thoe tflimib;EBSJQZh9pldxsq7'ce',WGN0crrhoHhyp9us&*UI'9 s s,aNgal ofjmlemn skVh UD?HUixtPfK1i3y, tcY3;]q4d OugalccXOnf GTbL0Xreof1K2-orken(1M2il abo,DNHM?h]
b"ug'SOxWayab."X]ThaDQ9"L&8o mevF; ugE uirdeasprd(UPelMFu

w_XQn lyn PkelY0SK9AcvC]d.,Ahy;CST4wblwvCRha s Lho&3.RithP(7zqWovCk-F*NMFnlQ4kR;-NDEFnW;drld727anv,6?72y:JFy7'4fL5F9OS?)Pewo3thowu"LJXWm,X8A6xQ&WilabFirjvL(H.I OGV0_D!EocrkYHgBhDY3g
GARk kJC0-e Du
liof
ziliss s63New:;DCKy'yad[(R]lJ_o
